# Comparative Study of Deep Learning Architectures for Khmer ASR
### Complete Google Colab Experiment & Retraining Pipeline

**Course**: Deep Learning Final Project (2026–2027)  
**Lecturer**: Mr. Soklong HIM  
**Hardware**: Google Colab Tesla T4 GPU  
**Dataset**: Google FLEURS Khmer (`km_kh`)  

### Saved approaches compared:
1. **Approach 1**: Whisper-Tiny full fine-tuning (Seq2Seq Transformer)
2. **Approach 2**: Meta MMS-1B partial fine-tuning with CTC

Both approaches use the seed-42 shuffled first 1,000 FLEURS training examples, the first 200 validation examples, and the first 200 test examples. The frozen-encoder experiment is excluded because its checkpoint and metrics are not available.

## 1. Hardware & GPU Check
Make sure your Colab session has a GPU assigned: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: Running on CPU. Please switch to T4 GPU in Runtime settings.')

## 2. Install Required Dependencies & Free Disk Space

In [ ]:
# Clean any old cached downloads to avoid filling Colab disk
!rm -rf /content/hf_cache /root/.cache/huggingface
!pip install -q torch torchaudio transformers datasets torchcodec evaluate jiwer accelerate tensorboard soundfile librosa matplotlib python-pptx

## 3. Clone Repository or Upload Code
If your repository is on GitHub, uncomment Option A. Otherwise, run Option B to upload your `src/`, `slides/`, and `samples/` folders.

In [ ]:
# Clone repository and enter directory
!git clone https://github.com/Seypa-47/khmer_asr.git
%cd khmer_asr


## 4. Run Approach 1: Whisper-Tiny Full Fine-Tuning (Matched Subset)
Trains `openai/whisper-tiny` on Google FLEURS `km_kh` (~1.5 GB). Uses `--skip-ddd` so it does NOT download the 60 GB DDD dataset, preventing disk quota errors.

In [ ]:
!python src/finetune_whisper.py \
  --output-dir "./models/whisper-tiny-khmer-reproduce" \
  --model-name "openai/whisper-tiny" \
  --use-fleurs-train \
  --max-train-samples 1000 \
  --max-eval-samples 200 \
  --max-test-samples 200 \
  --skip-ddd \
  --seed 42 \
  --num-train-epochs 3 \
  --learning-rate 1e-5 \
  --warmup-steps 35 \
  --per-device-train-batch-size 1 \
  --per-device-eval-batch-size 4 \
  --gradient-accumulation-steps 8 \
  --logging-steps 25 \
  --eval-steps 118 \
  --save-steps 118 \
  --fp16

## 5. Run Approach 2: Meta MMS-1B Khmer (CTC, Partial Fine-Tuning)
Unfreezes the top four Transformer encoder layers plus the CTC head/adapter. This is partial fine-tuning, not adapter-only training.

In [ ]:
# Train MMS with the same seed-42 shuffled 1,000-row training subset.
!python src/train_mms.py \
  --output-dir "./models/mms-khmer-ctc" \
  --model-id "facebook/mms-1b-all" \
  --target-lang "khm" \
  --seed 42 \
  --max-train-samples 1000 \
  --max-eval-samples 200 \
  --max-test-samples 200 \
  --num-train-epochs 15 \
  --learning-rate 5e-5 \
  --unfreeze-top-layers 4 \
  --apply-spec-augment \
  --lr-scheduler-type cosine \
  --warmup-steps 50 \
  --per-device-train-batch-size 1 \
  --per-device-eval-batch-size 1 \
  --gradient-accumulation-steps 8 \
  --eval-steps 50 \
  --save-steps 50 \
  --fp16


## 6. Optional frozen-encoder ablation (excluded from reported results)

The prior frozen-encoder checkpoint and its metrics are not available in this repository. Do not report an Approach 3 score unless the experiment is rerun and its checkpoint, predictions, metrics, and trainer state are saved.

## 7. Evaluate and generate results
Regenerates `results/learning_curves.png`, `results/metrics_comparison.png`, and `results/summary_table.md` from saved evidence. The checked-in PPTX must be reconciled with any changed scores before submission.

In [ ]:
# Re-score Whisper on the same held-out 200 examples and save predictions.
!python src/evaluate_saved_whisper.py --max-samples 200

# Generate results from saved test metrics and trainer state.
!python src/evaluate_and_plot.py

# The checked-in PPTX is the presentation deliverable. Update its results slide after any new run.

# Display figures directly in Colab
from IPython.display import Image, display
print('\n--- Learning Curves ---')
display(Image('results/learning_curves.png'))
print('\n--- Approaches CER Comparison ---')
display(Image('results/metrics_comparison.png'))

## 8. Package saved model, results, and slides
Packages the following files for download:
- `models/whisper-tiny-khmer/` (clean model weights ready for `app.py`)
- `results/` (figures and summary tables)
- `slides/` (PowerPoint presentation)

In [ ]:
# Clean heavy optimizer checkpoints
!rm -rf models/*/checkpoint-* models/*/runs 2>/dev/null || true
# Package all trained models (Whisper + MMS), results, and slides
!zip -r khmer_asr_trained_bundle.zip models/ results/ slides/
from google.colab import files
files.download('khmer_asr_trained_bundle.zip')
